# Sprint 4H Targeted Fine-Tuning

Thin Colab runner for targeted macro-F1 improvement. Core logic stays in `src/dl_midterm/` and `scripts/`. This run keeps full operational artifacts on Drive, including checkpoints, feature caches, and MLP `model.pt` files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -d dl-assignment/.git ]; then
  git clone https://github.com/KasimDeliaci/dl-midterm-project.git dl-assignment
fi
cd /content/dl-assignment
git pull --ff-only
python -m pip install -q uv
uv sync
mkdir -p /content/drive/MyDrive/dl-midterm-artifacts/sprint4h

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
BUNDLE=""
for candidate in \
  /content/drive/MyDrive/ham10000_colab_bundle.tar \
  "/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar" \
  /content/drive/MyDrive/dl-assignment/ham10000_colab_bundle.tar; do
  if [ -f "$candidate" ]; then BUNDLE="$candidate"; break; fi
done
if [ -z "$BUNDLE" ]; then
  echo "HAM10000 bundle not found on Drive" >&2
  exit 1
fi
tar -xf "$BUNDLE" -C /content/dl-assignment

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/finetune_backbone.py \
  --config configs/experiments/sprint4h_targeted_backbones.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_targeted \
  --batch-size 32

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/train_mlp.py \
  --config configs/experiments/sprint4h_targeted_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_targeted \
  --experiment-name sprint4h_targeted_single_backbone
uv run python scripts/run_experiment_matrix.py \
  --config configs/experiments/sprint4h_targeted_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_targeted \
  --experiment-name sprint4h_targeted_feature_matrix

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
DEST=/content/drive/MyDrive/dl-midterm-artifacts/sprint4h
mkdir -p \
  "$DEST/checkpoints/finetuned_targeted_backbones" \
  "$DEST/features/ham10000/finetuned_targeted" \
  "$DEST/runs" \
  "$DEST/report_assets"
rsync -a artifacts/checkpoints/finetuned_targeted_backbones/ \
  "$DEST/checkpoints/finetuned_targeted_backbones/"
rsync -a artifacts/features/ham10000/finetuned_targeted/ \
  "$DEST/features/ham10000/finetuned_targeted/"
rsync -a artifacts/runs/ \
  "$DEST/runs/"
rsync -a artifacts/report_assets/ \
  "$DEST/report_assets/"
echo "Backbone checkpoints on Drive:"
find "$DEST/checkpoints" -name '*.pt' | sort | wc -l
echo "Feature cache .pt files on Drive:"
find "$DEST/features" -name '*.pt' | sort | wc -l
echo "MLP model.pt files on Drive:"
find "$DEST/runs" -name 'model.pt' | sort | wc -l